# 02 — Silver Layer: 3NF with Enforced Data Quality

Builds `group1_finance.silver` from bronze. The design follows from `02_silver_profiling`.

**Model.** The 8 TPC-H entities in 3NF, with explicit types, primary keys and foreign keys — including the composite key `lineitem (l_partkey, l_suppkey) → partsupp (ps_partkey, ps_suppkey)`. ER diagram: [`docs/silver_er.md`](../docs/silver_er.md) (or *Catalog Explorer → silver → any table → View relationships*).

**How data quality is enforced** — three layers, because Unity Catalog does not enforce every constraint type:

| Constraint | Declared on the Delta table | Enforced by Delta on write | Enforced by this pipeline |
|---|---|---|---|
| Types | ✓ | ✓ | `try_cast`; a value that fails to cast is quarantined |
| `NOT NULL` | ✓ | ✓ | quarantined before write |
| `CHECK` (ranges, allowed values, positivity) | ✓ | ✓ | same expression quarantined before write |
| `PRIMARY KEY` | ✓ (informational) | ✗ | duplicate keys quarantined; uniqueness asserted after load |
| `FOREIGN KEY` | ✓ (informational) | ✗ | orphans quarantined (anti-join against the *clean* silver parent); asserted after load |

Every rule is evaluated per row. Rows that pass go to the silver table; rows that fail go to `silver.quarantine` with the list of rules they broke and the original bronze row as JSON — nothing is dropped silently.
Tables load parent-first, and FKs are checked against the already-cleaned parent, so quarantining a parent also quarantines its children (no orphans can reach silver).

**3NF notes.** TPC-H is already normalised: nation/region attributes live only in `nation`/`region`; the `partsupp` and `lineitem` non-key attributes depend on their whole composite key. Two columns are technically derivable and kept on purpose:
`o_totalprice` (≈ Σ line items) — it is the order's recorded total, and reconciling it against its line items is exactly the Finance validation rule; and `l_extendedprice` (= quantity × retail price) — it is the price at the time of sale, which would not survive a later retail-price change.

Prerequisites: `01_bronze` (and, for the rationale behind the bounds, `02_silver_profiling`).

In [0]:
# Parameters (override from a Job or the widget bar to run in another workspace)
dbutils.widgets.text("catalog", "group1_finance", "Catalog")
dbutils.widgets.text("bronze_schema", "bronze", "Bronze schema")
dbutils.widgets.text("silver_schema", "silver", "Silver schema")

CATALOG       = dbutils.widgets.get("catalog")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")
BRONZE        = f"{CATALOG}.{dbutils.widgets.get('bronze_schema')}"
SILVER        = f"{CATALOG}.{SILVER_SCHEMA}"

QUARANTINE = f"{SILVER}.quarantine"
LOAD_LOG   = f"{SILVER}.load_log"

from pyspark.sql import functions as F, Window

RUN_TS = spark.sql("SELECT current_timestamp() AS ts").first()["ts"]

print(f"Source : {BRONZE}")
print(f"Target : {SILVER}")
print(f"Run at : {RUN_TS}")

In [0]:
# Derived in 02_silver_profiling (section 8) - keep both notebooks in sync.
DISCOUNT_MIN, DISCOUNT_MAX = 0.00, 0.10
TAX_MIN, TAX_MAX           = 0.00, 0.08

MKT_SEGMENTS   = ["AUTOMOBILE", "BUILDING", "FURNITURE", "HOUSEHOLD", "MACHINERY"]
ORDER_STATUSES = ["F", "O", "P"]
RETURN_FLAGS   = ["A", "N", "R"]
LINE_STATUSES  = ["F", "O"]

def sql_in(values):
    return ", ".join(f"'{v}'" for v in values)

## Table definitions

One spec per table drives both the Delta DDL and the quarantine rules, so the constraint declared on the table and the rule applied to the data can never drift apart.

- `columns` — silver type; `NOT NULL` marks identity, relationship and finance-relevant columns. Descriptive text (names, addresses, comments) stays nullable on purpose: quarantining a customer for a missing phone number would cascade to all its orders and line items and distort revenue.
- `pk`, `fks` — declared as Unity Catalog constraints, enforced by the pipeline.
- `checks` — Delta `CHECK` constraints; a row passes when the expression is not `FALSE` (same semantics as Delta).

Tables are listed parent-first; that is the load order.

In [0]:
SILVER_TABLES = [
    {
        "name": "region",
        "comment": "TPC-H regions",
        "columns": {
            "r_regionkey": "BIGINT NOT NULL",
            "r_name":      "STRING NOT NULL",
            "r_comment":   "STRING",
        },
        "pk": ["r_regionkey"],
        "fks": [],
        "checks": {},
    },
    {
        "name": "nation",
        "comment": "TPC-H nations, each in one region",
        "columns": {
            "n_nationkey": "BIGINT NOT NULL",
            "n_name":      "STRING NOT NULL",
            "n_regionkey": "BIGINT NOT NULL",
            "n_comment":   "STRING",
        },
        "pk": ["n_nationkey"],
        "fks": [(["n_regionkey"], "region", ["r_regionkey"])],
        "checks": {},
    },
    {
        "name": "supplier",
        "comment": "Suppliers",
        "columns": {
            "s_suppkey":   "BIGINT NOT NULL",
            "s_name":      "STRING",
            "s_address":   "STRING",
            "s_nationkey": "BIGINT NOT NULL",
            "s_phone":     "STRING",
            "s_acctbal":   "DECIMAL(18,2)",   # may be negative - no range rule
            "s_comment":   "STRING",
        },
        "pk": ["s_suppkey"],
        "fks": [(["s_nationkey"], "nation", ["n_nationkey"])],
        "checks": {},
    },
    {
        "name": "customer",
        "comment": "Customers, including those with no orders",
        "columns": {
            "c_custkey":    "BIGINT NOT NULL",
            "c_name":       "STRING",
            "c_address":    "STRING",
            "c_nationkey":  "BIGINT NOT NULL",
            "c_phone":      "STRING",
            "c_acctbal":    "DECIMAL(18,2)",  # may be negative - no range rule
            "c_mktsegment": "STRING NOT NULL",
            "c_comment":    "STRING",
        },
        "pk": ["c_custkey"],
        "fks": [(["c_nationkey"], "nation", ["n_nationkey"])],
        "checks": {
            "c_mktsegment_valid": f"c_mktsegment IN ({sql_in(MKT_SEGMENTS)})",
        },
    },
    {
        "name": "part",
        "comment": "Parts (products)",
        "columns": {
            "p_partkey":     "BIGINT NOT NULL",
            "p_name":        "STRING",
            "p_mfgr":        "STRING",
            "p_brand":       "STRING",
            "p_type":        "STRING",
            "p_size":        "INT",
            "p_container":   "STRING",
            "p_retailprice": "DECIMAL(18,2) NOT NULL",
            "p_comment":     "STRING",
        },
        "pk": ["p_partkey"],
        "fks": [],
        "checks": {
            "p_retailprice_positive": "p_retailprice > 0",
        },
    },
    {
        "name": "partsupp",
        "comment": "Which supplier offers which part, at what cost",
        "columns": {
            "ps_partkey":    "BIGINT NOT NULL",
            "ps_suppkey":    "BIGINT NOT NULL",
            "ps_availqty":   "INT NOT NULL",
            "ps_supplycost": "DECIMAL(18,2) NOT NULL",
            "ps_comment":    "STRING",
        },
        "pk": ["ps_partkey", "ps_suppkey"],
        "fks": [
            (["ps_partkey"], "part",     ["p_partkey"]),
            (["ps_suppkey"], "supplier", ["s_suppkey"]),
        ],
        "checks": {
            "ps_supplycost_positive":   "ps_supplycost > 0",
            "ps_availqty_non_negative": "ps_availqty >= 0",
        },
    },
    {
        "name": "orders",
        "comment": "Order headers",
        "columns": {
            "o_orderkey":      "BIGINT NOT NULL",
            "o_custkey":       "BIGINT NOT NULL",
            "o_orderstatus":   "STRING NOT NULL",
            "o_totalprice":    "DECIMAL(18,2) NOT NULL",
            "o_orderdate":     "DATE NOT NULL",
            "o_orderpriority": "STRING",
            "o_clerk":         "STRING",
            "o_shippriority":  "INT",
            "o_comment":       "STRING",
        },
        "pk": ["o_orderkey"],
        "fks": [(["o_custkey"], "customer", ["c_custkey"])],
        "checks": {
            "o_totalprice_positive": "o_totalprice > 0",
            "o_orderstatus_valid":   f"o_orderstatus IN ({sql_in(ORDER_STATUSES)})",
        },
    },
    {
        "name": "lineitem",
        "comment": "Order lines; (l_partkey, l_suppkey) references partsupp",
        "columns": {
            "l_orderkey":      "BIGINT NOT NULL",
            "l_partkey":       "BIGINT NOT NULL",
            "l_suppkey":       "BIGINT NOT NULL",
            "l_linenumber":    "INT NOT NULL",
            "l_quantity":      "DECIMAL(18,2) NOT NULL",
            "l_extendedprice": "DECIMAL(18,2) NOT NULL",
            "l_discount":      "DECIMAL(18,2) NOT NULL",
            "l_tax":           "DECIMAL(18,2) NOT NULL",
            "l_returnflag":    "STRING NOT NULL",
            "l_linestatus":    "STRING NOT NULL",
            "l_shipdate":      "DATE NOT NULL",
            "l_commitdate":    "DATE NOT NULL",
            "l_receiptdate":   "DATE NOT NULL",
            "l_shipinstruct":  "STRING",
            "l_shipmode":      "STRING",
            "l_comment":       "STRING",
        },
        "pk": ["l_orderkey", "l_linenumber"],
        "fks": [
            (["l_orderkey"],              "orders",   ["o_orderkey"]),
            (["l_partkey", "l_suppkey"],  "partsupp", ["ps_partkey", "ps_suppkey"]),  # composite FK
        ],
        "checks": {
            "l_linenumber_positive":    "l_linenumber > 0",
            "l_quantity_positive":      "l_quantity > 0",
            "l_extendedprice_positive": "l_extendedprice > 0",
            "l_discount_range":         f"l_discount BETWEEN {DISCOUNT_MIN:.2f} AND {DISCOUNT_MAX:.2f}",
            "l_tax_range":              f"l_tax BETWEEN {TAX_MIN:.2f} AND {TAX_MAX:.2f}",
            "l_returnflag_valid":       f"l_returnflag IN ({sql_in(RETURN_FLAGS)})",
            "l_linestatus_valid":       f"l_linestatus IN ({sql_in(LINE_STATUSES)})",
        },
    },
]

SPECS = {s["name"]: s for s in SILVER_TABLES}
print("Load order:", " -> ".join(SPECS))

## Pipeline helpers

- `create_table` — DDL with types, `NOT NULL`, PK, FK, then `ALTER TABLE … ADD CONSTRAINT … CHECK`.
- `flag_rows` — evaluates every rule on a bronze DataFrame and adds `_failed_rules` (empty array = clean). Pure function, no writes; reused by the demo at the end.
- `load_table` — exact-duplicate rows are dropped (a load artefact, counted in `load_log`), then clean rows → silver, failing rows → quarantine.

In [0]:
def column_type(ddl):
    return ddl.replace("NOT NULL", "").strip()

def is_not_null(ddl):
    return ddl.endswith("NOT NULL")


def create_table(spec):
    name = spec["name"]
    defs = [f"{col} {ddl}" for col, ddl in spec["columns"].items()]
    defs.append(f"CONSTRAINT pk_{name} PRIMARY KEY ({', '.join(spec['pk'])})")
    for cols, parent, parent_cols in spec["fks"]:
        defs.append(
            f"CONSTRAINT fk_{name}_{parent} FOREIGN KEY ({', '.join(cols)}) "
            f"REFERENCES {SILVER}.{parent} ({', '.join(parent_cols)})"
        )
    spark.sql(
        f"CREATE TABLE {SILVER}.{name} (\n  " + ",\n  ".join(defs) + f"\n) COMMENT '{spec['comment']}'"
    )
    for rule, expr in spec["checks"].items():
        spark.sql(f"ALTER TABLE {SILVER}.{name} ADD CONSTRAINT {rule} CHECK ({expr})")


def dq_rules(spec):
    """(rule_name, SQL expression that is TRUE when the row passes). Expressions never evaluate to NULL."""
    rules = []
    for col, ddl in spec["columns"].items():
        rules.append((f"{col}_invalid_type", f"_raw_{col} IS NULL OR {col} IS NOT NULL"))
        if is_not_null(ddl):
            rules.append((f"{col}_not_null", f"{col} IS NOT NULL"))
    for rule, expr in spec["checks"].items():
        rules.append((rule, f"coalesce({expr}, true)"))  # Delta CHECK: only FALSE is a violation
    rules.append((f"pk_{spec['name']}_duplicate", "_pk_count = 1"))
    for _, parent, _ in spec["fks"]:
        rules.append((f"fk_{spec['name']}_{parent}_orphan", f"_fk_{parent} IS NOT NULL"))
    return rules


def flag_rows(spec, raw):
    columns = spec["columns"]
    df = raw.select(
        *[F.expr(f"try_cast({c} AS {column_type(ddl)})").alias(c) for c, ddl in columns.items()],
        *[F.col(c).alias(f"_raw_{c}") for c in columns],
        F.to_json(F.struct(*raw.columns), {"ignoreNullFields": "false"}).alias("_record"),
    )

    # Primary key: any key that occurs more than once is ambiguous -> every copy is quarantined
    df = df.withColumn("_pk_count", F.count(F.lit(1)).over(Window.partitionBy(*spec["pk"])))

    # Foreign keys: left join to the already-clean silver parent (unique on its PK, so no fan-out)
    for cols, parent, parent_cols in spec["fks"]:
        parent_keys = (
            spark.table(f"{SILVER}.{parent}")
            .select(*[F.col(p).alias(c) for c, p in zip(cols, parent_cols)])
            .withColumn(f"_fk_{parent}", F.lit(True))
        )
        df = df.join(parent_keys, on=cols, how="left")

    failed = [F.when(~F.expr(cond), F.lit(rule)) for rule, cond in dq_rules(spec)]
    return df.withColumn("_failed_rules", F.filter(F.array(*failed), lambda r: r.isNotNull()))


def load_table(spec):
    name, columns = spec["name"], list(spec["columns"])
    raw = spark.table(f"{BRONZE}.{name}")
    flagged = flag_rows(spec, raw.dropDuplicates())

    (flagged.filter(F.size("_failed_rules") == 0)
        .select(*columns)
        .write.mode("append").saveAsTable(f"{SILVER}.{name}"))

    (flagged.filter(F.size("_failed_rules") > 0)
        .select(
            F.lit(name).alias("source_table"),
            F.concat_ws("|", *[F.col(f"_raw_{c}").cast("string") for c in spec["pk"]]).alias("record_key"),
            F.col("_failed_rules").alias("failed_rules"),
            F.col("_record").alias("record"),
            F.lit(RUN_TS).alias("_quarantined_at"),
        )
        .write.mode("append").saveAsTable(QUARANTINE))

    bronze_rows = raw.count()
    silver_rows = spark.table(f"{SILVER}.{name}").count()
    quarantined = spark.table(QUARANTINE).filter(F.col("source_table") == name).count()
    return (RUN_TS, name, bronze_rows, silver_rows, quarantined, bronze_rows - silver_rows - quarantined)

In [0]:
# Full refresh: drop children before parents (FK references), then create parents before children.
for spec in reversed(SILVER_TABLES):
    spark.sql(f"DROP TABLE IF EXISTS {SILVER}.{spec['name']}")
spark.sql(f"DROP TABLE IF EXISTS {QUARANTINE}")

for spec in SILVER_TABLES:
    create_table(spec)
    print(f"Created {SILVER}.{spec['name']:<10s} pk={spec['pk']}  fks={len(spec['fks'])}  checks={len(spec['checks'])}")

spark.sql(f"""
CREATE TABLE {QUARANTINE} (
  source_table    STRING        NOT NULL COMMENT 'Silver table the row was meant for',
  record_key      STRING                 COMMENT 'Primary key value(s) from bronze, pipe-separated',
  failed_rules    ARRAY<STRING> NOT NULL COMMENT 'Names of the DQ rules the row broke',
  record          STRING        NOT NULL COMMENT 'Original bronze row as JSON',
  _quarantined_at TIMESTAMP     NOT NULL COMMENT 'Pipeline run timestamp'
) COMMENT 'Rows rejected by silver data-quality rules (rebuilt on every run)'
""")

# Append-only audit of every run, so row counts can be tracked over time
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {LOAD_LOG} (
  run_ts                 TIMESTAMP NOT NULL,
  table_name             STRING    NOT NULL,
  bronze_rows            BIGINT    NOT NULL,
  silver_rows            BIGINT    NOT NULL,
  quarantined_rows       BIGINT    NOT NULL,
  duplicate_rows_dropped BIGINT    NOT NULL
) COMMENT 'Silver load audit: bronze = silver + quarantined + exact duplicates'
""")
print(f"Created {QUARANTINE}, ensured {LOAD_LOG}")

In [0]:
log_rows = []
print(f"{'Table':<10s} {'Bronze':>12s} {'Silver':>12s} {'Quarantine':>11s} {'Dupes':>7s}")
print("-" * 56)
for spec in SILVER_TABLES:
    row = load_table(spec)
    log_rows.append(row)
    _, name, b, s, q, d = row
    print(f"{name:<10s} {b:>12,} {s:>12,} {q:>11,} {d:>7,}")

spark.createDataFrame(
    log_rows,
    "run_ts TIMESTAMP, table_name STRING, bronze_rows BIGINT, silver_rows BIGINT, quarantined_rows BIGINT, duplicate_rows_dropped BIGINT",
).write.mode("append").saveAsTable(LOAD_LOG)

## Validation

Delta already guarantees types, `NOT NULL` and `CHECK` on everything in silver (a violating write fails). The checks below cover what Unity Catalog does **not** enforce — primary-key uniqueness and foreign-key integrity — and summarise what was quarantined.
Cross-layer monetary reconciliation (bronze ↔ silver ↔ gold) and order-total checks live in the validation notebook.

In [0]:
q = spark.table(QUARANTINE)
if q.isEmpty():
    print("No rows quarantined - every bronze row passed all silver rules.")
else:
    display(
        q.select("source_table", F.explode("failed_rules").alias("rule"))
         .groupBy("source_table", "rule").count()
         .orderBy("source_table", F.desc("count"))
    )

In [0]:
for spec in SILVER_TABLES:
    df = spark.table(f"{SILVER}.{spec['name']}")
    r = df.agg(F.count("*").alias("rows"), F.countDistinct(*spec["pk"]).alias("keys")).first()
    status = "ok" if r["rows"] == r["keys"] else "DUPLICATES"
    print(f"{spec['name']:<10s} rows={r['rows']:>12,}  distinct pk={r['keys']:>12,}  {status}")
    assert r["rows"] == r["keys"], f"{spec['name']}: primary key is not unique"
print("\nAll primary keys unique.")

In [0]:
for spec in SILVER_TABLES:
    for cols, parent, parent_cols in spec["fks"]:
        parent_keys = spark.table(f"{SILVER}.{parent}").select(
            *[F.col(p).alias(c) for c, p in zip(cols, parent_cols)]
        )
        orphans = spark.table(f"{SILVER}.{spec['name']}").join(parent_keys, cols, "left_anti").count()
        print(f"{spec['name']:<9s} ({', '.join(cols)}) -> {parent} ({', '.join(parent_cols)}): orphans={orphans:,}")
        assert orphans == 0, f"fk_{spec['name']}_{parent}: {orphans} orphan rows"
print("\nAll foreign keys resolve.")

In [0]:
# PK / FK as registered in Unity Catalog (these drive the ER diagram in Catalog Explorer)
display(spark.sql(f"""
SELECT tc.table_name, tc.constraint_type, tc.constraint_name, kcu.column_name, kcu.ordinal_position
FROM {CATALOG}.information_schema.table_constraints tc
JOIN {CATALOG}.information_schema.key_column_usage kcu
  ON  tc.constraint_catalog = kcu.constraint_catalog
  AND tc.constraint_schema  = kcu.constraint_schema
  AND tc.constraint_name    = kcu.constraint_name
WHERE tc.table_schema = '{SILVER_SCHEMA}'
ORDER BY tc.table_name, tc.constraint_type DESC, tc.constraint_name, kcu.ordinal_position
"""))

# CHECK constraints are stored as Delta table properties
check_rows = []
for spec in SILVER_TABLES:
    for p in spark.sql(f"SHOW TBLPROPERTIES {SILVER}.{spec['name']}").collect():
        if p["key"].startswith("delta.constraints."):
            check_rows.append((spec["name"], p["key"].removeprefix("delta.constraints."), p["value"]))
display(spark.createDataFrame(check_rows, "table STRING, check_constraint STRING, expression STRING"))

## Demo: data quality in action

Two short demonstrations for the presentation. Neither leaves anything behind in silver.

1. **Delta rejects a bad write.** We try to append a line item with a 50% discount directly to `silver.lineitem`, bypassing the pipeline.
2. **The pipeline routes bad rows to quarantine.** We take a few real bronze line items, corrupt them on purpose, and run them through `flag_rows` to see which rules catch them.

In [0]:
bad_row = (
    spark.table(f"{SILVER}.lineitem").limit(1)
    .withColumn("l_discount", F.lit("0.50").cast("decimal(18,2)"))
)

try:
    bad_row.write.mode("append").saveAsTable(f"{SILVER}.lineitem")
    rejected, message = False, ""
except Exception as e:
    rejected, message = True, str(e)

if not rejected:  # should never happen - clean up so silver stays valid
    spark.sql(f"DELETE FROM {SILVER}.lineitem WHERE l_discount = 0.50")
assert rejected, "CHECK constraint l_discount_range did not fire"
print("Write rejected by Delta, as expected:\n")
print(message.splitlines()[0][:400])

In [0]:
w = Window.orderBy("l_orderkey", "l_linenumber")
sample = (
    spark.table(f"{BRONZE}.lineitem").filter("l_orderkey <= 10")
    .withColumn("_n", F.row_number().over(w)).filter("_n <= 6")
)

def corrupt(n, col, value):
    return F.when(F.col("_n") == n, value).otherwise(F.col(col)).alias(col)

corrupted = sample.select(
    "_n",
    corrupt(5, "l_orderkey", F.lit(-1)),                                # orphan: order does not exist
    "l_partkey",
    corrupt(6, "l_suppkey", F.col("l_suppkey") + 1),                    # (part, supplier) pair not in partsupp
    *[c for c in SPECS["lineitem"]["columns"]
      if c not in ("l_orderkey", "l_suppkey", "l_partkey", "l_quantity", "l_discount", "l_tax")],
    corrupt(3, "l_quantity", F.lit("-5").cast("decimal(18,2)")),        # non-positive quantity
    corrupt(2, "l_discount", F.lit("0.50").cast("decimal(18,2)")),      # discount out of range
    corrupt(4, "l_tax", F.lit(None).cast("decimal(18,2)")),             # missing tax
)
# Row 1 stays clean. Row 2 gets a conflicting copy (same key, different comment) -> one row can break several rules
corrupted = corrupted.unionByName(
    corrupted.filter("_n = 2").withColumn("_n", F.lit(7)).withColumn("l_comment", F.lit("conflicting copy"))
)

display(
    flag_rows(SPECS["lineitem"], corrupted.drop("_n"))
    .select("l_orderkey", "l_linenumber", "l_partkey", "l_suppkey", "l_quantity", "l_discount", "l_tax", "_failed_rules")
    .orderBy("l_orderkey", "l_linenumber")
)